Нотбук выполняет этап классификации постов на основе ранее обученной модели:
1. Загрузка собранных сырых данных `data_raw.csv`.
2. Загрузка оптимального бинарного классификатора `best_binary_classifier.joblib`.
3. Получение вероятностей принадлежности к классам.
4. Расчет симметричного скора уверенности (Signed Score) в диапазоне от -1.0 до +1.0.
5. Сегментация постов по обновленным симметричным градациям уверенности (Very Strong Vacancy, Vacancy, Uncertain, Resume, Very Strong Resume).
6. Визуализация распределения полученных скоров.
7. Экспорт результатов классификации в `data_classified.csv`.

In [ ]:
import os
import joblib
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Настройки графиков
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

In [ ]:
DATA_DIR = "data"
MODELS_DIR = "models"

RAW_DATA_PATH = os.path.join(DATA_DIR, "data_raw.csv")
MODEL_PATH = os.path.join(MODELS_DIR, "best_binary_classifier.joblib")
CLASSIFIED_DATA_PATH = os.path.join(DATA_DIR, "data_classified.csv")

# Проверка наличия обязательных файлов
if not os.path.exists(RAW_DATA_PATH):
    raise FileNotFoundError(f"Файл сырых данных {RAW_DATA_PATH} не найден!")

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Файл обученной модели {MODEL_PATH} не найден!")

# Загрузка
df_raw = pd.read_csv(RAW_DATA_PATH)
best_model = joblib.load(MODEL_PATH)

print(f"Загружено собранных постов для классификации: {len(df_raw)}")
print("Обученная модель классификации успешно импортирована.")

In [ ]:
# Получение предсказания вероятностей
probs_raw = best_model.predict_proba(df_raw["text"].fillna("").astype(str).values)
df_raw["p_vacancy"] = probs_raw[:, 1]

# Масштабирование вероятности [0.0, 1.0] в симметричную шкалу signed_score [-1.0, +1.0]
df_raw["signed_score"] = 2 * df_raw["p_vacancy"] - 1

print("Расчет вероятностей и signed_score успешно завершен.")

In [ ]:
def get_confidence_tier(score: float) -> str:
    """
    Группирует непрерывный signed_score [-1.0, 1.0] в симметричные качественные градации.
    
    Границы классов:
    - score >= 0.9           -> Very Strong Vacancy
    - 0.7 <= score < 0.9     -> Vacancy
    - -0.7 < score < 0.7     -> Uncertain
    - -0.9 < score <= -0.7   -> Resume
    - score <= -0.9          -> Very Strong Resume
    """
    if score >= 0.90:
        return "Very Strong Vacancy"
    elif score >= 0.70:
        return "Vacancy"
    elif score <= -0.90:
        return "Very Strong Resume"
    elif score <= -0.70:
        return "Resume"
    else:
        return "Uncertain"

# Применение классификации уверенности
df_raw["confidence_tier"] = df_raw["signed_score"].apply(get_confidence_tier)

print("Категоризация постов по классам уверенности завершена.")

In [ ]:
print("=" * 60)
print("РАСПРЕДЕЛЕНИЕ ПОЛУЧЕННЫХ КЛАССОВ УВЕРЕННОСТИ")
print("=" * 60)
tier_counts = df_raw["confidence_tier"].value_counts()
print(tier_counts)

# Отрисовка гистограммы распределения скоров
plt.figure(figsize=(10, 6))
sns.histplot(df_raw["signed_score"], bins=50, kde=True, color="purple")

# Границы классов уверенности на графике
plt.axvline(x=0.90, color="red", linestyle="--", label="Very Strong Vacancy (>= 0.90)")
plt.axvline(x=0.70, color="orange", linestyle="--", label="Vacancy (>= 0.70)")
plt.axvline(x=-0.70, color="blue", linestyle="--", label="Resume (<= -0.70)")
plt.axvline(x=-0.90, color="darkblue", linestyle="--", label="Very Strong Resume (<= -0.90)")

plt.title("Распределение подписанных скоров (Signed Score)")
plt.xlabel("Signed Score")
plt.ylabel("Количество постов")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("ПРИМЕРЫ ПОСТОВ ИЗ КАЖДОЙ КАТЕГОРИИ УВЕРЕННОСТИ")
print("=" * 60)

categories = ["Very Strong Vacancy", "Vacancy", "Uncertain", "Resume", "Very Strong Resume"]

for tier in categories:
    subset = df_raw[df_raw["confidence_tier"] == tier]
    print(f"\nКатегория: {tier.upper()} (Всего найдено: {len(subset)} постов)")
    print("-" * 55)
    
    if len(subset) > 0:
        # Случайный выбор одного примера для демонстрации стабильности
        sample = subset.sample(1, random_state=42).iloc[0]
        print(f"ID: {sample['id']} | Канал: {sample['channel']}")
        print(f"P(Vacancy): {sample['p_vacancy']:.4f} | Signed Score: {sample['signed_score']:.4f}")
        print("Текст оригинального сообщения (первые 300 символов):")
        print(str(sample["original_text"])[:300] + "...")
    else:
        print("В текущей выборке посты этой категории отсутствуют.")

In [ ]:
# Финальный набор колонок для экспорта в соответствии со стандартами обработки данных
export_columns = [
    "id", 
    "channel", 
    "date", 
    "original_text", 
    "text", 
    "text_hash", 
    "p_vacancy", 
    "signed_score", 
    "confidence_tier"
]

df_export = df_raw[export_columns].copy()

# Сохранение в data_classified.csv
df_export.to_csv(CLASSIFIED_DATA_PATH, index=False, encoding="utf-8-sig")

print("=" * 60)
print(f"Классифицированные посты сохранены в: {CLASSIFIED_DATA_PATH}")
print(f"Размерность итоговой таблицы: {df_export.shape}")
print("=" * 60)